In [2]:
# Cell 1: Define inputs, outputs, and your genomic region of interest
import os

In [4]:
bed_v3 = "primers/v3/nCoV-2019.insert.bed"
bed_v4 = "primers/v4/SARS-CoV-2.insert.bed"
output_bed = "spike_v3_v4_overlaps.bed"

#spike protein
target_region = "NC_045512.2:21492-25259" 

# Parse the target region coordinates
t_chrom, coords = target_region.split(":")
t_start, t_end = map(int, coords.split("-"))

In [5]:
# Cell 2: Helper function to parse BED files and filter by region
def parse_and_filter(filepath, t_chrom, t_start, t_end):
    data = {}
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith(('#', 'track', 'browser')) or not line.strip():
                continue
            parts = line.strip().split('\t')
            chrom, start, end, name = parts[0], int(parts[1]), int(parts[2]), parts[3]
            
            # Extract numerical ID from the name (e.g., "SARS-CoV-2_INSERT_71" or "71" -> "71")
            amplicon_id = name.split('_')[-1] 
            
            # Keep amplicons falling within your specified region window
            if chrom == t_chrom and (t_start <= start <= t_end or t_start <= end <= t_end):
                data[amplicon_id] = {
                    'chrom': chrom,
                    'start': start,
                    'end': end,
                    'name': name
                }
    return data

# Load datasets
v3_data = parse_and_filter(bed_v3, t_chrom, t_start, t_end)
v4_data = parse_and_filter(bed_v4, t_chrom, t_start, t_end)

print(f"Loaded {len(v3_data)} amplicons from V3 and {len(v4_data)} from V4 within region.")

Loaded 13 amplicons from V3 and 14 from V4 within region.


In [6]:
# Cell 3: Calculate overlaps and save each to an individual BED file in the current directory
common_ids = sorted(list(set(v3_data.keys()) & set(v4_data.keys())), key=int)
saved_files_count = 0

for amp_id in common_ids:
    a = v3_data[amp_id]
    b = v4_data[amp_id]
    
    # Mathematical overlap boundaries
    overlap_start = max(a['start'], b['start'])
    overlap_end = min(a['end'], b['end'])
    
    # Check if a physical overlap exists
    if overlap_start < overlap_end:
        overlap_length = overlap_end - overlap_start
        filename = f"amp_{amp_id}_overlap.bed"
        
        # Write individual BED entry directly to current working directory
        with open(filename, 'w') as out:
            # Format: chrom, start, end, name, overlap_length
            out.write(f"{a['chrom']}\t{overlap_start}\t{overlap_end}\tamp_{amp_id}_overlap\t{overlap_length}\n")
        
        saved_files_count += 1

print(f"Successfully generated {saved_files_count} separate BED files in the current working directory.")

Successfully generated 13 separate BED files in the current working directory.
